In [ ]:
from pyspark.sql.functions import *

In [ ]:
df_bronze=(
    spark
    .readStream
    .format('delta')
    .table("bronze.API_Raw_Data")
)

In [ ]:
df_valid=df_bronze.filter(trim(col('results'))!='[]')

In [ ]:
df_normalized=df_valid.withColumn('results',
                when(
                    trim(col('results')).startswith("["),
                    trim(col('results'))).
                otherwise(
                    concat(lit('['),trim(col('results')),
                    lit(']')
                    )
                )
            )

In [ ]:
df_silver=df_normalized.select(
        get_json_object(col('results'),"$[0].gender").alias('Gender'),
        get_json_object(col('results'),"$[0].name.title").alias('Title'),
        get_json_object(col('results'),"$[0].name.first").alias('First'),
        get_json_object(col('results'),"$[0].name.last").alias('Last'),
        concat(get_json_object(col('results'),"$[0].location.street.number"),lit(', '),get_json_object(col('results'),"$[0].location.street.name")).alias('Street_Info'),
        get_json_object(col('results'),"$[0].location.city").alias('City'),
        get_json_object(col('results'),"$[0].location.state").alias('State'),
        get_json_object(col('results'),"$[0].location.country").alias('Country'),
        get_json_object(col('results'),"$[0].location.postcode").alias('Postcode'),
        get_json_object(col('results'),"$[0].location.coordinates.latitude").alias('Latitude'),
        get_json_object(col('results'),"$[0].location.coordinates.longitude").alias('Longitude'),
        get_json_object(col('results'),"$[0].location.timezone.offset").alias('TimeZone_Offset'),
        get_json_object(col('results'),"$[0].location.timezone.description").alias('TimeZone_Description'),
        get_json_object(col('results'),"$[0].email").alias('Email'),
        get_json_object(col('results'),"$[0].login.uuid").alias('userid'),
        get_json_object(col('results'),"$[0].login.username").alias('User_Name'),
        get_json_object(col('results'),"$[0].login.password").alias('Password'),
        get_json_object(col('results'),"$[0].login.salt").alias('Salt'),
        get_json_object(col('results'),"$[0].login.md5").alias('MD5'),
        get_json_object(col('results'),"$[0].login.sha1").alias('SHA1'),
        get_json_object(col('results'),"$[0].login.sha256").alias('SHA256'),
        get_json_object(col('results'),"$[0].dob.date").alias('Birth_Date'),
        get_json_object(col('results'),"$[0].dob.age").alias('Age'),
        get_json_object(col('results'),"$[0].registered.date").alias('Registered_Date'),
        get_json_object(col('results'),"$[0].registered.age").alias('Registered_Age'),
        get_json_object(col('results'),"$[0].phone").alias('Phone_Number'),
        get_json_object(col('results'),"$[0].cell").alias('Cell_Number'),
        get_json_object(col('results'),"$[0].id.name").alias('ID_Name'),
        get_json_object(col('results'),"$[0].id.value").alias('ID_Value'),
        get_json_object(col('results'),"$[0].picture.large").alias('Picture_Large'),
        get_json_object(col('results'),"$[0].picture.medium").alias('Picture_Medium'),
        get_json_object(col('results'),"$[0].picture.thumbnail").alias('Picture_thumbnail'),
        get_json_object(col('results'),"$[0].nat").alias('NAT'),
        col('injestion_timestamp').alias('Injestion_Timestamp'),
        current_timestamp().alias("Processing_Timestamp")

        )

In [ ]:
query=(df_silver.writeStream
                .format('delta')
                .outputMode("append")
                .option('checkpointLocation',
                        'Files/checkpoint/api_silver')
                .toTable("Silver.API_Silver_Data")
    )
query.awaitTermination()

In [1]:
content='''
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark=SparkSession.builder.getOrCreate()
df_bronze=(
    spark
    .readStream
    .format('delta')
    .table("bronze.API_Raw_Data")
)
df_valid=df_bronze.filter(trim(col('results'))!='[]')
df_normalized=df_valid.withColumn('results',
                when(
                    trim(col('results')).startswith("["),
                    trim(col('results'))).
                otherwise(
                    concat(lit('['),trim(col('results')),
                    lit(']')
                    )
                )
            )
df_silver=df_normalized.select(
        get_json_object(col('results'),"$[0].gender").alias('Gender'),
        get_json_object(col('results'),"$[0].name.title").alias('Title'),
        get_json_object(col('results'),"$[0].name.first").alias('First'),
        get_json_object(col('results'),"$[0].name.last").alias('Last'),
        concat(get_json_object(col('results'),"$[0].location.street.number"),lit(', '),get_json_object(col('results'),"$[0].location.street.name")).alias('Street_Info'),
        get_json_object(col('results'),"$[0].location.city").alias('City'),
        get_json_object(col('results'),"$[0].location.state").alias('State'),
        get_json_object(col('results'),"$[0].location.country").alias('Country'),
        get_json_object(col('results'),"$[0].location.postcode").alias('Postcode'),
        get_json_object(col('results'),"$[0].location.coordinates.latitude").alias('Latitude'),
        get_json_object(col('results'),"$[0].location.coordinates.longitude").alias('Longitude'),
        get_json_object(col('results'),"$[0].location.timezone.offset").alias('TimeZone_Offset'),
        get_json_object(col('results'),"$[0].location.timezone.description").alias('TimeZone_Description'),
        get_json_object(col('results'),"$[0].email").alias('Email'),
        get_json_object(col('results'),"$[0].login.uuid").alias('userid'),
        get_json_object(col('results'),"$[0].login.username").alias('User_Name'),
        get_json_object(col('results'),"$[0].login.password").alias('Password'),
        get_json_object(col('results'),"$[0].login.salt").alias('Salt'),
        get_json_object(col('results'),"$[0].login.md5").alias('MD5'),
        get_json_object(col('results'),"$[0].login.sha1").alias('SHA1'),
        get_json_object(col('results'),"$[0].login.sha256").alias('SHA256'),
        get_json_object(col('results'),"$[0].dob.date").alias('Birth_Date'),
        get_json_object(col('results'),"$[0].dob.age").alias('Age'),
        get_json_object(col('results'),"$[0].registered.date").alias('Registered_Date'),
        get_json_object(col('results'),"$[0].registered.age").alias('Registered_Age'),
        get_json_object(col('results'),"$[0].phone").alias('Phone_Number'),
        get_json_object(col('results'),"$[0].cell").alias('Cell_Number'),
        get_json_object(col('results'),"$[0].id.name").alias('ID_Name'),
        get_json_object(col('results'),"$[0].id.value").alias('ID_Value'),
        get_json_object(col('results'),"$[0].picture.large").alias('Picture_Large'),
        get_json_object(col('results'),"$[0].picture.medium").alias('Picture_Medium'),
        get_json_object(col('results'),"$[0].picture.thumbnail").alias('Picture_thumbnail'),
        get_json_object(col('results'),"$[0].nat").alias('NAT'),
        col('injestion_timestamp').alias('Injestion_Timestamp'),
        current_timestamp().alias("Processing_Timestamp")

        )
query=(df_silver.writeStream
                .format('delta')
                .outputMode("append")
                .option('checkpointLocation',
                        'Files/checkpoint/api_silver')
                .toTable("Silver.API_Silver_Data")
    )
query.awaitTermination()
'''
notebookutils.fs.put("Files/bronze_to_silver.py",content,True)
print('Production Python file created succesfully')


StatementMeta(, 49d49233-c3c8-48f1-b369-3495d783b1b0, 3, Finished, Available, Finished, False)

Production Python file created succesfully


In [2]:
print(notebookutils.fs.ls("Files/"))

StatementMeta(, 483c0546-0d41-483f-8b51-d0ecff9a263e, 4, Finished, Available, Finished, False)

[FileInfo(path=abfss://87af2989-49a5-40e9-960f-2da392c0bac9@onelake.dfs.fabric.microsoft.com/d037bf72-b0fa-4785-92a9-84df50cd03e7/Files/bronze_to_silver.py, name=bronze_to_silver.py, size=3703), FileInfo(path=abfss://87af2989-49a5-40e9-960f-2da392c0bac9@onelake.dfs.fabric.microsoft.com/d037bf72-b0fa-4785-92a9-84df50cd03e7/Files/checkpoint, name=checkpoint, size=0), FileInfo(path=abfss://87af2989-49a5-40e9-960f-2da392c0bac9@onelake.dfs.fabric.microsoft.com/d037bf72-b0fa-4785-92a9-84df50cd03e7/Files/checkpoints, name=checkpoints, size=0)]


In [2]:
print(notebookutils.fs.ls("Files/"))

StatementMeta(, 49d49233-c3c8-48f1-b369-3495d783b1b0, 4, Finished, Available, Finished, False)

[FileInfo(path=abfss://87af2989-49a5-40e9-960f-2da392c0bac9@onelake.dfs.fabric.microsoft.com/d037bf72-b0fa-4785-92a9-84df50cd03e7/Files/bronze_to_silver.py, name=bronze_to_silver.py, size=3781), FileInfo(path=abfss://87af2989-49a5-40e9-960f-2da392c0bac9@onelake.dfs.fabric.microsoft.com/d037bf72-b0fa-4785-92a9-84df50cd03e7/Files/checkpoint, name=checkpoint, size=0), FileInfo(path=abfss://87af2989-49a5-40e9-960f-2da392c0bac9@onelake.dfs.fabric.microsoft.com/d037bf72-b0fa-4785-92a9-84df50cd03e7/Files/checkpoints, name=checkpoints, size=0)]
